
# <font color="green">A Generic Optimizer for Any → Float Functions</font>

* Extend the previous problem `parametric/optimize_float` so that a single function definition can work for any function $f$ taking a parameter of an arbitrary type and returning a floating-point number ($T \to \mathbb{R}$).
* The crux is how to parameterize the input type of $f$.
* To that end, define a type `domain` parameterized by a type parameter $T$, whose `next` method returns a value of type $T$.
  * Depending on the language, `domain` may be an *abstract* type (interface, trait, etc.).
* Then define a generic `optimize` function that takes a function $f : T \to \mathbb{R}$ and a domain $D$ of type $T$, and finds an approximation of $\min_{x \in D} f(x)$ by evaluating $f(x)$ for each $x$ returned by $D\texttt{.next()}$ until the sentinel value is returned.
* Make the range type from the previous problem conform to `domain`, so that the generic optimizer can accept it.
* Define a type representing a 2D range and a constructor $\texttt{arange2d}(x_0, x_1, m, y_0, y_1, n)$, and make it conform to `domain` as well.
  * $\texttt{arange2d}(x_0, x_1, m, y_0, y_1, n)$ should generate $mn$ points $(x_0 + i\,\Delta x,\ y_0 + j\,\Delta y)$ for all combinations of $0 \leq i < m$ and $0 \leq j < n$, where $\Delta x = (x_1 - x_0)/(m-1)$ and $\Delta y = (y_1 - y_0)/(n-1)$.
  * To represent a 2D point, use:
    * a 2-element float array in Go (expression: `[2]float64{x, y}`, type: `[2]float64`)
    * a 2-float tuple in Julia (expression: `(x, y)`)
    * a 2-float tuple in OCaml (expression: `(x, y)`, type: `(float * float)`)
    * a 2-float tuple in Rust (expression: `(x, y)`, type: `(f64, f64)`)
* You may be amazed by the fact that in some languages, the float-only version you implemented in the previous problem and the generic version you define here is identical
* Boilerplate source files `{go,jl,ml,rs}/generic_optimize.{go,jl,ml,rs}` containing test code are generated and shown below.
* Edit the source files by opening them in a text editor (e.g., VS Code) or by editing and executing the cells below.




# 1. AI tutor
## 1-1. Prepare
* Your personal AI tutor is provided for questions and feedback
* Execute the following cell before you use it

In [ ]:
import heytutor

## 1-2. Examples
### 1-2-1. A general question
```
%%hey
How to write a function in Go?
```

### 1-2-2. A hint on this specific problem
```
%%hey
Give me a hint on this problem for Rust
```

### 1-2-3. <font color="red">NEW:</font> A few builtin variables
* `{file:FILENAME}` is the content of FILE
* `{bash[-1]}` is the output of the last `%%bash_` cell, `{bash[-2]}` that of the second last `%%bash_` cell, etc.
* `{problem}` is the content of the file you specified by `%%hey problem_file=foo.md`
* `{answer}` is the content of the file you specified by `%%hey answer_file=go/foo.go`

### 1-2-4. Help when you struggle
```
%%hey answer_file=go/foo.go
I get this error when I compile it. What's wrong?"

My program:
{answer}

Error message:
{bash[-1]}

```

### 1-2-5. Ask feedback
* You are encouraged to ask a feedback once you think you are done with the problem, to know if there is a better answer.  You can do so by something like:

```
%%hey problem_file=foo.md answer_file=go/foo.md
Give me a feedback to my answer.

Problem:
{problem}

My Answer:
{answer}
```

# 2. Go

## 2-1. Baseline code

In [ ]:
import heytutor

In [ ]:
%%writefile_ go/generic_optimize.go
package main
import (
	"fmt"
	"math"
)

/** begin my answer */

type Domain[T any] interface {
	next() (bool, T)
}

type Arange struct {
	a float64
	b float64
	i int
	n int
	dx float64
}

func arange(a, b float64, n int) Arange {
	return Arange{a, b, 0, n, (b - a) / float64(n - 1)}
}

func (a *Arange) next() (bool, float64) {
	if a.i == a.n {
		return false, 0.0
	} else {
		x := a.a + a.dx * float64(a.i)
		a.i += 1
		return true, x
	}
}

type Arange2d struct {
	x0 float64
	x1 float64
	m int
	y0 float64
	y1 float64
	n int
	i int
	j int
	dx float64
	dy float64
}

func arange2d(x0, x1 float64, m int, y0, y1 float64, n int) Arange2d {
	return Arange2d{x0, x1, m, y0, y1, n, 0, 0, (x1 - x0) / float64(m - 1), (y1 - y0) / float64(n - 1)}
}

func (a *Arange2d) next() (bool, [2]float64) {
	if a.i == a.m {
		return false, [2]float64{0.0, 0.0}
	} else {
		x := a.x0 + a.dx * float64(a.i)
		y := a.y0 + a.dy * float64(a.j)
		if a.j + 1 == a.n {
			a.i += 1
			a.j = 0
		} else {
			a.j += 1
		}
		return true, [2]float64{x, y}
	}
}

func optimize[S any](f func(S) float64, rng Domain[S]) (bool, float64) {
	var miny float64
	ok, x := rng.next()
	i := 0
	for ok {
		y := f(x)
		if i == 0 || y < miny {
			miny = y
		}
		i += 1
		ok, x = rng.next()
	}
	return i > 0, miny
}

/** end my answer */
func very_close(x, y float64) bool {
	return math.Abs(x - y) < 1e-6
}

func main() {
	a0 := arange(0, 1, 10000)
	a1 := arange(0, 3, 10000)
	a2 := arange2d(-2, 2, 10000, -2, 2, 10000)
	f0 := func (x float64) float64 { return x * (x - 1) }
	f1 := func (x float64) float64 { return x * (x - 1) * (x - 2) }
	f2 := func (x [2]float64) float64 { return x[0] * x[0] + 2 * x[1] * x[1] + 3 * x[0] * x[1] }
	if ok, y0 := optimize(f0, &a0); !(ok && very_close(y0, -0.25))     { panic("wrong") }
	if ok, y1 := optimize(f1, &a1); !(ok && very_close(y1, -0.384900)) { panic("wrong") }
	if ok, y2 := optimize(f2, &a2); !(ok && very_close(y2, -0.5))      { panic("wrong") }
	fmt.Println("OK")
}

## 2-2. Compile

In [ ]:
%%bash_
export PATH=${PATH}:~/.local/go/bin:~/go/bin
go build -o go/generic_optimize go/generic_optimize.go

* Note: when you run `go` or other Go commands in a terminal (SSH or Jupyter terminal), you need to execute the first line (`export PATH=${PATH}:~/go/bin`)
* You may consider adding that line in your `~/.bash_profile`

## 2-3. Run

In [ ]:
%%bash_
go/generic_optimize

## 2-4. Ask Questions or Get Feedback

In [ ]:
%%hey problem_file=generic_optimize.md answer_file=go/generic_optimize.go

Problem:
{problem}
My Answer (between /** begin my answer */ and /** end my answer */):
{answer}

Give me a feedback to my answer.

# 3. Julia

## 3-1. Baseline code

In [ ]:
import heytutor

In [ ]:
%%writefile_ jl/generic_optimize.jl
### begin my answer

mutable struct Arange
    a::Float64
    b::Float64
    i::Int
    n::Int
    dx::Float64
end

function arange(a, b, n)
    Arange(a, b, 0, n, (b - a) / (n - 1))
end

function next(a::Arange)
    if a.i == a.n
        nothing
    else
        x = a.a + a.dx * a.i
        a.i += 1
        x
    end
end

mutable struct Arange2d
    x0::Float64
    x1::Float64
    m::Int
    y0::Float64
    y1::Float64
    n::Int
    i::Int
    j::Int
    dx::Float64
    dy::Float64
end

function arange2d(x0, x1, m, y0, y1, n)
    dx = (x1 - x0) / (m - 1)
    dy = (y1 - y0) / (n - 1)
    Arange2d(x0, x1, m, y0, y1, n, 0, 0, dx, dy)
end

function next(a::Arange2d)
    if a.i == a.m
        nothing
    else
        x = a.x0 + a.dx * a.i
        y = a.y0 + a.dy * a.j
        if a.j + 1 == a.n
            a.i += 1
            a.j = 0
        else
            a.j += 1
        end
        (x, y)
    end
end

function optimize(f, rng)
    minx, miny = nothing, nothing
    x = next(rng)
    while x != nothing
        y = f(x)
        if miny == nothing || y < miny
            minx, miny = x, y
        end
        x = next(rng)
    end
    miny
end

### end my answer

very_close(x, y) = abs(x - y) < 1e-6

function main()
    a0 = arange(0, 1, 10000)
    a1 = arange(0, 3, 10000)
    a2 = arange2d(-2, 2, 10000, -2, 2, 1000)
    f0(x) = x * (x - 1)
    f1(x) = x * (x - 1) * (x - 2)
    f2((x, y)) = x * x + 2 * y * y + 3 * x * y
    y0 = optimize(f0, a0)
    y1 = optimize(f1, a1)
    y2 = optimize(f2, a2)
    @assert(very_close(y0, -0.25))
    @assert(very_close(y1, -0.384900))
    @assert(very_close(y2, -0.5))
    println("OK")
end

main()

## 3-2. Compile

* Julia code is compiled "just in time" (compiled upon executed), so does not need a specific action for compilation before you run

## 3-3. Run

In [ ]:
%%bash_
export PATH=${PATH}:~/.juliaup/bin
julia jl/generic_optimize.jl

* Note: when you run `julia` or other Julia commands in a terminal (SSH or Jupyter terminal), you need to execute the first line (`export PATH=${PATH}:~/.juliaup/bin`)
* You may consider adding that line in your `~/.bash_profile`

## 3-4. Interactive execution
* `julia` command  also serves is an interactive command for Julia programs

* You can run a source code and continue interaction

```
$ julia -i jl/generic_optimize.jl
```

* For trial and error, you may also consider creating a Julia notebook


## 3-5. Ask Questions or Get Feedback

In [ ]:
%%hey problem_file=generic_optimize.md answer_file=jl/generic_optimize.jl

Problem:
{problem}

My Answer (between ### begin my answer and ### end my answer):
{answer}

Give me a feedback to my answer.

# 4. OCaml

## 4-1. Baseline code

In [ ]:
import heytutor

In [ ]:
%%writefile_ ml/generic_optimize.ml
(** begin my answer *)

let arange a b n =
  let dx = (b -. a) /. float_of_int (n - 1) in
  object 
    val mutable i = 0
    method next =
      if i = n then None
      else
        let x = a +. dx *. float_of_int i in
        let _ = i <- i + 1 in
        Some x
  end

let arange2d x0 x1 m y0 y1 n =
  let dx = (x1 -. x0) /. float_of_int (m - 1) in
  let dy = (y1 -. y0) /. float_of_int (n - 1) in
  object 
    val mutable i = 0
    val mutable j = 0
    method next =
      if i = m then
        None
      else 
        let x = x0 +. dx *. float_of_int i in
        let y = y0 +. dy *. float_of_int j in
        let _ = i <- if j + 1 = n then i + 1 else i in
        let _ = j <- if j + 1 = n then 0 else j + 1 in
        Some (x, y)
  end

let optimize f rng =
  let rec iter x miny =
    match (x, miny) with
    | (None, _) -> miny
    | (Some x, None) ->
       iter rng#next (Some(f x))
    | (Some x, Some(miny)) ->
       let y = f x in
       let next = rng#next in
       if y < miny then
         iter next (Some(y))
       else
         iter next (Some(miny))
  in
  iter (rng#next) None
(** end my answer *)

let very_close a b =
  match a with
  | None -> false
  | Some a -> Float.abs (a -. b) < 1e-6

let main () =
  let f0 x = x *. (x -. 1.) in
  let f1 x = x *. (x -. 1.) *. (x -. 2.) in
  let f2 (x, y) = x *. x +. 2. *. y *. y +. 3. *. x *. y in
  let a0 = arange (-1.) 3. 10000 in
  let a1 = arange (-0.) 3. 10000 in
  let a2 = arange2d (-2.) 2. 10000 (-2.) 2. 1000 in
  let y0 = optimize f0 a0 in
  let y1 = optimize f1 a1 in
  let y2 = optimize f2 a2 in
  assert (very_close y0 (-0.25));
  assert (very_close y1 (-0.384900));
  assert (very_close y2 (-0.5));

  Printf.printf "OK\n"
;;

main()

## 4-2. Compile

In [ ]:
%%bash_
eval $(opam env)
ocamlc ml/generic_optimize.ml -o ml/generic_optimize

* Note: when you run `ocamlc` or other OCaml commands (see below) in a terminal (SSH or Jupyter terminal), you need to execute the first line (`eval $(opam env)`)
* You may consider adding that line in your `~/.bash_profile`

## 4-3. Run

In [ ]:
%%bash_
ml/generic_optimize

## 4-4. Interactive execution
* `ocaml` command is an interactive command for OCaml programs

* In terminal (Jupyter or SSH), you can directly run a source code

```
$ eval $(opam env)   # once in your session or put it in ~/.bash_profile
$ ocaml ml/generic_optimize.ml
```

* You can run a source code and continue interaction

```
$ eval $(opam env)   # once in your session or put it in ~/.bash_profile
$ ocaml -init ml/generic_optimize.ml
```

* For trial and error, you may also consider creating an OCaml notebook


## 4-5. Ask Questions or Get Feedback

In [ ]:
%%hey problem_file=generic_optimize.md answer_file=ml/generic_optimize.ml

Problem:
{problem}

My Answer (between (** begin my answer *) and (** end my answer *)):
{answer}

Give me a feedback to my answer.

# 5. Rust

## 5-1. Baseline code

In [ ]:
import heytutor

In [ ]:
%%writefile_ rs/generic_optimize.rs
/** begin my answer */

trait Domain<T> {
    fn next(self : &mut Self) -> Option<T>;
}

#[allow(dead_code)]
struct Arange {
    a : f64,
    b : f64,
    i : i64,
    n : i64,
    dx : f64
}

fn arange(a : f64, b : f64, n : i64) -> Arange {
    Arange{a, b, i: 0, n, dx: (b - a) / ((n - 1) as f64)}
}

impl Domain<f64> for Arange {
    fn next(self : &mut Arange) -> Option<f64> {
	if self.i == self.n {
	    None
	} else {
	    let x = self.a + self.dx * (self.i as f64);
	    self.i += 1;
	    Some(x)
	}
    }
}

#[allow(dead_code)]
struct Arange2d {
    x0 : f64,
    x1 : f64,
    m  : i64,
    y0 : f64,
    y1 : f64,
    n  : i64,
    i  : i64,
    j  : i64,
    dx : f64,
    dy : f64
}

fn arange2d(x0 : f64, x1 : f64, m : i64, y0 : f64, y1 : f64, n : i64) -> Arange2d {
    Arange2d{x0, x1, m, y0, y1, n, i: 0, j: 0, dx: (x1 - x0) / ((m - 1) as f64), dy: (y1 - y0) / ((n - 1) as f64)}
}

impl Domain<(f64, f64)> for Arange2d {
    fn next(self : &mut Arange2d) -> Option<(f64, f64)> {
	if self.i == self.m {
	    None
	} else {
	    let x = self.x0 + self.dx * (self.i as f64);
	    let y = self.y0 + self.dy * (self.j as f64);
	    if self.j + 1 == self.n {
		self.i += 1;
		self.j = 0;
	    } else {
		self.j += 1;
	    }
	    Some((x, y))
	}
    }
}

fn optimize<S : Clone>(f : fn(S) -> f64, dom : &mut dyn Domain<S>) -> Option<f64> {
    match dom.next() {
	None => None,
	Some(x) => {
	    let mut miny = f(x);
	    while let Some(x) = dom.next() {
		let y = f(x);
		if y < miny {
		    miny = y;
		}
	    }
	    Some(miny)
	}
    }
}
/** end my answer */

fn very_close(a : Option<f64>, b : f64) -> bool {
    match a {
	None => false,
	Some(a) => (a - b).abs() < 1e-6
    }
}

fn main() {
    fn f0(x : f64) -> f64 { x * (x - 1.) }
    fn f1(x : f64) -> f64 { x * (x - 1.) * (x - 2.) }
    fn f2((x, y) : (f64, f64)) -> f64 { x * x + 2. * y * y + 3. * x * y }
    let mut a0 = arange(0., 1., 10000);
    let mut a1 = arange(0., 3., 10000);
    let mut a2 = arange2d(-2., 2., 10000, -2., 2., 1000);
    let y0 = optimize(f0, &mut a0);
    let y1 = optimize(f1, &mut a1);
    let y2 = optimize(f2, &mut a2);
    very_close(y0, -0.25);
    very_close(y1, -0.384900);
    very_close(y2, -0.5);
    println!("OK")
}

## 5-2. Compile

In [ ]:
%%bash_
. ~/.cargo/env
rustc rs/generic_optimize.rs -o rs/generic_optimize

* Note: when you run `rustc` or other Rust commands in a terminal (SSH or Jupyter terminal), you need to execute the first line (`. ~/.cargo/env`)
* You may consider adding that line in your `~/.bash_profile`

## 5-3. Run

In [ ]:
%%bash_
rs/generic_optimize

## 5-4. Ask Questions or Get Feedback

In [ ]:
%%hey problem_file=generic_optimize.md answer_file=rs/generic_optimize.rs

Problem:
{problem}

My Answer (between /** begin my answer */ and /** end my answer */):
{answer}

Give me a feedback to my answer.